# Environment Setup

This section imports the required libraries and initializes project configuration.

In [1]:
# Import libraries and configure project settings

import warnings
warnings.filterwarnings("ignore")

import json
import joblib
import boto3
import sagemaker
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

import matplotlib.pyplot as plt

RANDOM_STATE = 42

PROJECT_PREFIX = "loanwise"

sess = sagemaker.Session()
bucket = sess.default_bucket()

print(f"SageMaker Version : {sagemaker.__version__}")
print(f"Bucket            : {bucket}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


SageMaker Version : 2.224.4
Bucket            : sagemaker-us-east-1-612256167011


# Load Training Artifacts

Load the processed training and testing datasets created in Notebook 1.

In [2]:
# Load processed datasets

train_df = pd.read_csv("processed_train.csv")
test_df = pd.read_csv("processed_test.csv")

print(f"Training Shape : {train_df.shape}")
print(f"Testing Shape  : {test_df.shape}")

display(train_df.head())

Training Shape : (16000, 18)
Testing Shape  : (4000, 18)


,SK_ID_CURR,TARGET,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,DAYS_BIRTH,DAYS_EMPLOYED,CNT_FAM_MEMBERS,REGION_POPULATION_RELATIVE,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,CODE_GENDER_M,FLAG_OWN_CAR_Y,FLAG_OWN_REALTY_Y,event_time
0,125849,1,0,67500.0,425133.0,16024.5,351000.0,-19180,-8989,2.0,0.015221,0.628374,0.285300,0.468660,0,0,1,2026-06-13T11:39:45Z
1,425101,0,0,405000.0,675000.0,54252.0,675000.0,-9986,-568,1.0,0.046220,0.440128,0.658954,0.202087,1,0,1,2026-06-13T11:39:45Z
2,191342,0,0,180000.0,1125000.0,32895.0,1125000.0,-16639,-2253,1.0,0.072508,0.440128,0.787716,0.342529,0,0,0,2026-06-13T11:39:45Z
3,177438,1,0,382500.0,790830.0,62613.0,675000.0,-10254,-166,1.0,0.072508,0.114901,0.464413,0.151935,1,1,1,2026-06-13T11:39:45Z
4,141285,1,0,112500.0,343800.0,13090.5,225000.0,-14803,-1469,2.0,0.018029,0.440128,0.559160,0.144648,1,0,1,2026-06-13T11:39:45Z


# Prepare Modeling Data

Separate features and target variables.

In [3]:
# Prepare feature matrices and target vectors

TARGET_COLUMN = "TARGET"

drop_columns = [
    "TARGET",
    "event_time"
]

X_train = train_df.drop(columns=drop_columns)

y_train = train_df[TARGET_COLUMN]

X_test = test_df.drop(columns=drop_columns)

y_test = test_df[TARGET_COLUMN]

print(f"Training Features : {X_train.shape}")
print(f"Testing Features  : {X_test.shape}")

Training Features : (16000, 16)
Testing Features  : (4000, 16)


# Train Logistic Regression

Train the benchmark classification model.

In [4]:
# Train Logistic Regression model

logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_STATE
)

logistic_model.fit(
    X_train,
    y_train
)

print("Logistic Regression training completed.")

Logistic Regression training completed.


# Evaluate Logistic Regression

Evaluate benchmark model performance.

In [5]:
# Evaluate Logistic Regression

lr_predictions = logistic_model.predict(X_test)

lr_metrics = {
    "Accuracy": accuracy_score(y_test, lr_predictions),
    "Precision": precision_score(y_test, lr_predictions),
    "Recall": recall_score(y_test, lr_predictions),
    "F1": f1_score(y_test, lr_predictions),
    "ROC_AUC": roc_auc_score(y_test, lr_predictions)
}

display(pd.DataFrame([lr_metrics]))

,Accuracy,Precision,Recall,F1,ROC_AUC
0,0.6645,0.673158,0.6395,0.655897,0.6645


# Train Random Forest and XGBoost

Train advanced classification models.

In [6]:
# Train Random Forest and XGBoost

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf_model.fit(
    X_train,
    y_train
)

xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=RANDOM_STATE,
    eval_metric="logloss"
)

xgb_model.fit(
    X_train,
    y_train
)

print("Random Forest training completed.")
print("XGBoost training completed.")

Random Forest training completed.
XGBoost training completed.


# Evaluate Random Forest and XGBoost

Measure performance using classification metrics.

In [7]:
# Evaluate Random Forest and XGBoost

rf_predictions = rf_model.predict(X_test)

xgb_predictions = xgb_model.predict(X_test)

rf_metrics = {
    "Model": "Random Forest",
    "Accuracy": accuracy_score(y_test, rf_predictions),
    "Precision": precision_score(y_test, rf_predictions),
    "Recall": recall_score(y_test, rf_predictions),
    "F1": f1_score(y_test, rf_predictions),
    "ROC_AUC": roc_auc_score(y_test, rf_predictions)
}

xgb_metrics = {
    "Model": "XGBoost",
    "Accuracy": accuracy_score(y_test, xgb_predictions),
    "Precision": precision_score(y_test, xgb_predictions),
    "Recall": recall_score(y_test, xgb_predictions),
    "F1": f1_score(y_test, xgb_predictions),
    "ROC_AUC": roc_auc_score(y_test, xgb_predictions)
}

display(pd.DataFrame([
    rf_metrics,
    xgb_metrics
]))

,Model,Accuracy,Precision,Recall,F1,ROC_AUC
0,Random Forest,0.67175,0.678627,0.6525,0.665307,0.67175
1,XGBoost,0.67125,0.672024,0.6690,0.670509,0.67125


# Compare Models

Compare benchmark and advanced models and select the best model.

In [8]:
# Compare model performance

comparison_df = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        **lr_metrics
    },
    rf_metrics,
    xgb_metrics
])

comparison_df = comparison_df.sort_values(
    by="F1",
    ascending=False
)

display(comparison_df)

best_model_name = comparison_df.iloc[0]["Model"]

print(f"Best Model : {best_model_name}")

,Model,Accuracy,Precision,Recall,F1,ROC_AUC
2,XGBoost,0.67125,0.672024,0.6690,0.670509,0.67125
1,Random Forest,0.67175,0.678627,0.6525,0.665307,0.67175
0,Logistic Regression,0.66450,0.673158,0.6395,0.655897,0.66450


Best Model : XGBoost


# Save and Upload Artifacts

Save trained models, evaluation metrics, and upload artifacts to Amazon S3.

In [9]:
# Save model artifacts

joblib.dump(
    logistic_model,
    "logistic_regression_model.joblib"
)

joblib.dump(
    rf_model,
    "random_forest_model.joblib"
)

joblib.dump(
    xgb_model,
    "xgboost_model.joblib"
)

if best_model_name == "Logistic Regression":
    best_model = logistic_model

elif best_model_name == "Random Forest":
    best_model = rf_model

else:
    best_model = xgb_model

joblib.dump(
    best_model,
    "best_model.joblib"
)

comparison_df.to_json(
    "evaluation_metrics.json",
    orient="records",
    indent=4
)

best_model_s3 = sess.upload_data(
    path="best_model.joblib",
    bucket=bucket,
    key_prefix=f"{PROJECT_PREFIX}/models"
)

metrics_s3 = sess.upload_data(
    path="evaluation_metrics.json",
    bucket=bucket,
    key_prefix=f"{PROJECT_PREFIX}/metrics"
)

print(best_model_s3)
print(metrics_s3)

s3://sagemaker-us-east-1-612256167011/loanwise/models/best_model.joblib
s3://sagemaker-us-east-1-612256167011/loanwise/metrics/evaluation_metrics.json


# Verify Generated Artifacts

Review generated files and model performance outputs.

In [10]:
# Verify generated artifacts

print("Generated Local Artifacts")
print("-------------------------")
print("logistic_regression_model.joblib")
print("random_forest_model.joblib")
print("xgboost_model.joblib")
print("best_model.joblib")
print("evaluation_metrics.json")

print("\nS3 Artifacts")
print("------------")
print(best_model_s3)
print(metrics_s3)

print("\nFinal Model Ranking")
display(comparison_df)

Generated Local Artifacts
-------------------------
logistic_regression_model.joblib
random_forest_model.joblib
xgboost_model.joblib
best_model.joblib
evaluation_metrics.json

S3 Artifacts
------------
s3://sagemaker-us-east-1-612256167011/loanwise/models/best_model.joblib
s3://sagemaker-us-east-1-612256167011/loanwise/metrics/evaluation_metrics.json

Final Model Ranking


,Model,Accuracy,Precision,Recall,F1,ROC_AUC
2,XGBoost,0.67125,0.672024,0.6690,0.670509,0.67125
1,Random Forest,0.67175,0.678627,0.6525,0.665307,0.67175
0,Logistic Regression,0.66450,0.673158,0.6395,0.655897,0.66450
